# Domain Data And Fine-Tuning Readiness

Fine-tuning data is not the same thing as retrieval data. Documents are useful as evidence at answer time; supervised examples teach behavior. Fine-tuning tools expect structured examples, such as chat-format records, not only raw documents.


## Learning Objectives

By the end of this notebook, you should be able to:

- Separate retrieval documents, behavior examples, and evaluation requests.
- Explain how mixing data roles causes leakage.
- Validate style examples as chat records with Pydantic.
- Check behavior examples for fine-tuning readiness.


## Three Data Roles, One Job Each

Data readiness is less about having many files and more about assigning each record a job. The same raw domain material should not be pushed into every strategy:

| Role | What it is | Used for | Leakage risk |
|---|---|---|---|
| **Retrieval corpus** | Source-of-truth documents the system is allowed to search: authoritative, chunkable, and traceable to a source | Evidence the model cites at answer time | Style examples added to it are retrieved as if they were policy |
| **Behavior examples** | Input-output pairs that show the desired answer, tone, structure, or refusal pattern | Few-shot prompts and supervised fine-tuning records | Treated as evidence, they spread unofficial or outdated facts |
| **Evaluation set** | Held-out requests, including easy cases, edge cases, and cases where the system should abstain | Measuring whether a strategy works | Reused as training examples, they make the measurement too optimistic |

In this repo, the policy documents are the corpus, the style examples are the behavior examples, and the support requests are the evaluation set. A behavior example is the record shape supervised fine-tuning trains on, but it should not be treated as evidence unless it also points to an authoritative source. The evaluation set stays separate from both: it is the measuring stick for whether a strategy works.

## Load Shared Data And Runtime Helpers


In [ ]:
import json
from pprint import pprint

from pydantic import BaseModel, Field, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    call_model,
)

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | Model: {CONFIG.model} | Dry run: {CONFIG.dry_run}"
)

## Classify Data Roles

Count the records in each role before preparing any of them.

In [ ]:
inventory = [
    {"name": "policy documents", "count": len(DOCUMENTS), "role": "retrieval_corpus"},
    {
        "name": "style examples",
        "count": len(STYLE_EXAMPLES),
        "role": "behavior_examples",
    },
    {"name": "support requests", "count": len(REQUESTS), "role": "evaluation_requests"},
]
pprint(inventory)

## What Makes An Example Useful?

A useful behavior example is specific enough to teach the desired response pattern without smuggling in unstable facts. Weak examples are often too broad: “Reply politely” or “be helpful” does not teach the model when to refuse, when to ask a clarifying question, or when to cite policy. “Please send the invoice number and both payment references.” is more useful because it shows the exact reply, with its structure and tone.

Before a record becomes a few-shot example or future training example, we check four things:

1. **Realistic input:** the user message is realistic enough to match a future request.
2. **Exact behavior:** the assistant answer demonstrates the exact behavior we want repeated.
3. **Labels:** the record has labels such as intent or tone, so the dataset can be audited.
4. **No unstable facts:** the answer avoids hardcoding unstable facts unless they are also present in a source document. If a detail such as “finance replies within three business days” changes often, it belongs in retrieval evidence rather than in a behavior-only example.

Across records, examples also need consistency. If one refund example says to ask for invoice numbers and another says to approve refunds immediately, the dataset teaches conflicting behavior. This notebook develops the inspection checks needed before records become prompts or supervised examples; the supplied data remains intentionally smaller than a production training set.

## Convert Style Examples Into Chat Records

The same message shape works for few-shot prompting today and can be adapted into future supervised fine-tuning JSONL records.


Define small validation models before converting any examples. The schema catches unknown roles or empty messages early.

In [ ]:
class ChatMessage(BaseModel):
    """A minimal chat message record for fine-tuning-style examples."""

    role: str = Field(pattern="^(system|user|assistant)$")
    content: str = Field(min_length=1)


class TrainingExample(BaseModel):
    """Provider-neutral training example with labels for auditability."""

    messages: list[ChatMessage]
    intent: str
    tone: str

Check that the schema actually rejects a broken record: a role that chat formats do not know and an empty message.

In [ ]:
try:
    ChatMessage(role="customer", content="")
except ValidationError as error:
    print(error)

Convert each style example into the same chat-message shape used by supervised fine-tuning tools.


In [ ]:
def to_training_example(example: dict[str, str]) -> TrainingExample:
    """Convert one style example into a chat-shaped record."""
    return TrainingExample(
        intent=example["intent"],
        tone=example["tone"],
        messages=[
            ChatMessage(
                role="system",
                content=f"Answer as a support assistant. Tone: {example['tone']}.",
            ),
            ChatMessage(role="user", content=example["user"]),
            ChatMessage(role="assistant", content=example["assistant"]),
        ],
    )

Build the converted records and inspect one example before using them in readiness checks.


In [ ]:
training_examples = [to_training_example(example) for example in STYLE_EXAMPLES]
pprint(training_examples[0].model_dump())

Look at the assistant message. Besides the response pattern (ask for the invoice number and both payment references), it also states a policy fact: finance normally replies within three business days. The same fact is in the `refund-policy` document, so the example is correct today. But if finance changes the deadline, the policy document gets updated while this example keeps teaching the old one. This is the unstable-fact check from the checklist above, applied to our own data.

## Ask The Model To Review Readiness

The model can help critique whether examples are ready for future fine-tuning. This is a live provider call when credentials exist; dry mode returns a deterministic review.


Serialize the validated examples so the provider reviews the same structure we inspected.


In [ ]:
readiness_prompt = json.dumps(
    [example.model_dump() for example in training_examples], indent=2
)

readiness_messages = [
    {
        "role": "system",
        "content": (
            "You review supervised fine-tuning datasets. Return a JSON object with exactly "
            "three keys: strengths, gaps, and next_steps. Each value is a list of short strings."
        ),
    },
    {
        "role": "user",
        "content": f"Review these examples for future fine-tuning readiness:\n{readiness_prompt}",
    },
]

Keep the dry review deterministic so the lesson can run without credentials.


In [ ]:
dry_readiness = json.dumps(
    {
        "strengths": [
            "examples are chat-shaped",
            "intent and tone labels are explicit",
        ],
        "gaps": [
            "too few examples",
            "billing intent is missing",
            "no held-out style evaluation set",
        ],
        "next_steps": [
            "collect more examples per intent",
            "separate training and evaluation records",
        ],
    },
    indent=2,
)

Call the provider or dry-mode fallback, then validate the review. Asking for JSON does not guarantee that every requested key comes back, so a small Pydantic model checks the shape the same way it checked the chat records.

In [ ]:
class ReadinessReview(BaseModel):
    """The review shape requested in the system prompt."""

    strengths: list[str]
    gaps: list[str]
    next_steps: list[str]


review_reply = call_model(
    readiness_messages,
    dry_response=dry_readiness,
    response_format={"type": "json_object"},
)
print(f"Review ({review_reply.source}):")

try:
    review = ReadinessReview.model_validate_json(review_reply.text)
    pprint(review.model_dump())
except ValidationError as error:
    print("The review does not match the requested shape:")
    print(error)

### Exercise: Find Uncovered Intents

A model review can name real gaps, miss them, or list generic ones the data cannot confirm. Check one possible gap against the data yourself: return the sorted list of intents that appear in the evaluation requests (`REQUESTS`) but have no style example (`STYLE_EXAMPLES`).

In [ ]:
# @TODO: Compare the intents in REQUESTS and STYLE_EXAMPLES.


<details>
<summary>Show Reference Solution</summary>

```python
style_intents = {example["intent"] for example in STYLE_EXAMPLES}
request_intents = {request["intent"] for request in REQUESTS}
uncovered_intents = sorted(request_intents - style_intents)
uncovered_intents
```

</details>


Only `billing` is uncovered: the request about correcting an invoice address has no example of the desired response behavior. This gap is concrete and verifiable in the data. Compare it with the review above: if the review missed it, or listed gaps the data cannot confirm, that is why we treat a model review as a starting point and check its claims.

## Check Your Understanding

1. Which records should stay held out: retrieval documents, style examples, or evaluation requests?
2. What does each strategy (retrieval, supervised fine-tuning, and evaluation) need from the raw domain material?
3. Why should style examples not be added to the retrieval corpus?
4. The refund style example tells the customer that finance normally replies within three business days. If that deadline changes often, where should this detail live instead, and what should the example keep?

<details>
<summary>Show Reference Answers</summary>

1. Evaluation requests should stay held out so they can measure generalization and strategy quality.
2. Retrieval needs source documents with metadata, supervised fine-tuning needs input-output examples, and evaluation needs held-out requests.
3. The retriever could return an old example answer as if it were source-of-truth policy, so the model would cite unofficial or outdated facts. Style examples teach behavior; policy documents provide evidence.
4. Only in the `refund-policy` document, which is retrieved at answer time and updated when the deadline changes. The example should keep the response pattern: ask for the invoice number and both payment references. Otherwise a model prompted or trained with the example keeps repeating the old deadline.

</details>


## Further Reading

- [LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206) - Zhou et al., 2023
- [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) - Gebru et al., 2018
